In [4]:
import copy
import json

In [3]:
def deep_merge(target, source):
    """
    Deep merges source into target.
    For lists (arrays), the source list overwrites the target list.
    For dicts (objects), keys are recursively merged.
    
    Args:
        target: The base dictionary or list.
        source: The overriding dictionary or list.
        visible_keys: A list/set of keys to retain in the resulting dictionary.
    """
    # If either is not a dict/list, return the source (override) or target
    if not isinstance(target, (dict, list)) or target is None:
        return source if source is not None else target
    if not isinstance(source, (dict, list)) or source is None:
        return target

    # Handle arrays (lists): overwrite rather than concatenate, as in the JS version
    if isinstance(target, list) and isinstance(source, list):
        return copy.deepcopy(source)

    # Handle objects (dicts): deep merge
    if isinstance(target, dict) and isinstance(source, dict):
        output = copy.deepcopy(target)

        for key, source_value in source.items():
            target_value = output.get(key)

            if isinstance(target_value, list) and isinstance(source_value, list):
                output[key] = copy.deepcopy(source_value)
            elif isinstance(target_value, dict) and isinstance(source_value, dict):
                output[key] = deep_merge(target_value, source_value)
            else:
                output[key] = copy.deepcopy(source_value)

        return output

    # Fallback for mismatched types
    return copy.deepcopy(source)

In [1]:


def _extract_refs(node, found_refs):
    """Recursively search for $ref values in a dictionary or list."""
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "$ref" and isinstance(value, str) and value.startswith("#/$defs/"):
                # Extract the definition name from the path
                def_name = value.split("/")[-1]
                found_refs.add(def_name)
            else:
                _extract_refs(value, found_refs)
    elif isinstance(node, list):
        for item in node:
            _extract_refs(item, found_refs)

def reduce_schema(schema, visible=True):
    """
    Reduces the schema by filtering properties to only those in visibleSections,
    and pruning $defs to only those referenced by the visible properties.
    """
    if not visible:
        return copy.deepcopy(schema)

    reduced = copy.deepcopy(schema)
    visible_sections = reduced.get("visibleSections", [])
    
    # 1. Filter properties based on the visibleSections list
    original_properties = reduced.get("properties", {})
    filtered_properties = {
        k: v for k, v in original_properties.items() if k in visible_sections
    }
    reduced["properties"] = filtered_properties

    # 2. Find all $defs referenced directly by the filtered properties
    required_defs = set()
    _extract_refs(filtered_properties, required_defs)

    # 3. Recursively find $defs referenced within other required $defs
    original_defs = reduced.get("$defs", {})
    processed_defs = set()
    
    while required_defs - processed_defs:
        current_def = (required_defs - processed_defs).pop()
        processed_defs.add(current_def)
        
        if current_def in original_defs:
            _extract_refs(original_defs[current_def], required_defs)

    # 4. Filter the $defs dictionary to only include the required ones
    reduced["$defs"] = {
        k: v for k, v in original_defs.items() if k in required_defs
    }

    # 5. Clean-up: filter root 'required' array so it only lists visible properties
    if "required" in reduced and isinstance(reduced["required"], list):
        reduced["required"] = [
            req for req in reduced["required"] if req in filtered_properties
        ]

    return reduced

In [5]:
directory = "../src/utils/"
with open(f"{directory}HDRUK4.0.0.json") as h:
    hdruk = json.load(h)
with open(f"{directory}CRUK1.0.0.json") as c:
    cruk = json.load(c)
with open(f"{directory}semanticSchema.json") as s:
    semantic = json.load(s)

In [6]:
mid = deep_merge(hdruk, cruk)
final = deep_merge(mid, semantic)

In [9]:
merged = reduce_schema(final)

In [10]:
with open(f"{directory}merged.json", "w") as f:
    json.dump(merged, f, indent=3)